In [1]:
!pip install pdf2image

In [2]:
!apt-get install poppler-utils # Install poppler-

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 49 not upgraded.
Need to get 186 kB of archives.
After this operation, 696 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.5 [186 kB]
Fetched 186 kB in 0s (1,825 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 123632 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.5_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.5) ...
Setting up poppler-utils (22.02.0-2ubuntu0.5) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
from huggingface_hub import hf_hub_download
from transformers import AutoImageProcessor, TableTransformerForObjectDetection
import torch
from PIL import Image
from pdf2image import convert_from_path # Import to convert PDF to image

# file_path = '/content/acs.jafc.7b03597 split pea.pdf'
file_path = '/content/1-s2.0-S0308814617312839-Lentils.pdf'
# Convert the PDF pages to a list of PIL Images
images = convert_from_path(file_path)

# Initialize the processor and model
image_processor = AutoImageProcessor.from_pretrained("microsoft/table-transformer-detection")
model = TableTransformerForObjectDetection.from_pretrained("microsoft/table-transformer-detection")

# Loop over all pages in the PDF
for page_num, image in enumerate(images):
    print(f"Processing page {page_num + 1}")

    # Process the image
    inputs = image_processor(images=image, return_tensors="pt")
    outputs = model(**inputs)

    # Convert outputs (bounding boxes and class logits) to Pascal VOC format (xmin, ymin, xmax, ymax)
    target_sizes = torch.tensor([image.size[::-1]])  # Ensure image dimensions are in the correct order
    results = image_processor.post_process_object_detection(outputs, threshold=0.8, target_sizes=target_sizes)[0]

    # Loop through the detected tables and save or process the bounding boxes
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        print(
            f"Detected {model.config.id2label[label.item()]} with confidence "
            f"{round(score.item(), 3)} at location {box}"
        )

        # Crop the detected table from the image based on bounding box
        xmin, ymin, xmax, ymax = map(int, box)
        cropped_table = image.crop((xmin, ymin, xmax, ymax))

        # Optionally, save the cropped table as an image
        cropped_table.save(f"extracted_table_page_{page_num + 1}.png")

        # You can also apply OCR to extract text from the table image if needed


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Some weights of the model checkpoint at microsoft/table-transformer-detection were not used when initializing TableTransformerForObjectDetection: ['model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TableTransformerForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Processing page 1
Detected table with confidence 0.974 at location [108.06, 928.55, 1545.47, 1185.17]
Processing page 2
Processing page 3
Detected table with confidence 1.0 at location [121.86, 1676.35, 1536.69, 1977.99]
Processing page 4
Detected table with confidence 1.0 at location [121.03, 235.44, 1535.53, 491.09]
Detected table with confidence 1.0 at location [121.12, 1651.97, 1531.48, 1906.58]
Processing page 5
Detected table with confidence 1.0 at location [120.9, 234.46, 1535.83, 492.45]
Processing page 6


| Protein    | Food Type       | %DM   | %CF  | %CP   | ASP  | THR  | SER  | GLU   | PRO   | GLY  | ALA  | CYS  | VAL  | MET  | ILE  | LEU   | TYR  | PHE  | HIS  | LYS  | ARG  | TRP  |
|------------|-----------------|-------|------|-------|------|------|------|-------|-------|------|------|------|------|------|------|-------|------|------|------|------|------|------|
| Casein     | -               | 93.36 | 0.21 | 8.32  | 3.58 | 0.85 | 1.25 | 21.43 | 10.44 | 1.44 | 3.38 | 0.83 | 5.37 | 1.02 | 0.87  | 1.55  | 5.16 | 4.91 | 2.93 | 7.44 | 3.33 | 1.15 |
| Yellow Pea | Untreated       | 91.30 | 1.50 | 23.78 | 2.88 | 0.91 | 1.30 | 3.94  | 0.96  | 0.92 | 1.13 | 0.25 | 1.12 | 0.22 | 1.75  | 4.10  | 0.62 | 1.04 | 0.65 | 1.68 | 2.02 | 0.21 |
| Yellow Pea | Extruded        | 95.62 | 1.53 | 24.56 | 3.13 | 0.87 | 1.32 | 4.20  | 0.96  | 0.99 | 1.20 | 0.26 | 1.10 | 0.23 | 1.90  | 1.75  | 0.70 | 1.17 | 0.69 | 1.75 | 2.14 | 0.20 |
| Yellow Pea | Cooked          | 97.44 | 2.91 | 22.88 | 2.85 | 0.87 | 1.30 | 3.84  | 0.91  | 0.88 | 1.15 | 0.23 | 1.10 | 0.21 | 1.97  | 1.18  | 0.67 | 1.18 | 0.64 | 1.73 | 2.06 | 0.19 |
| Yellow Pea | Baked           | 95.81 | 2.16 | 23.35 | 2.92 | 0.88 | 1.27 | 4.07  | 0.82  | 0.97 | 1.20 | 0.25 | 1.12 | 0.21 | 1.87  | 1.15  | 0.63 | 1.15 | 0.68 | 1.61 | 2.00 | 0.22 |
| Green Pea  | Untreated       | 91.43 | 0.35 | 26.15 | 3.16 | 0.93 | 1.37 | 4.46  | 1.01  | 1.02 | 1.25 | 0.31 | 1.14 | 0.24 | 1.96  | 0.74  | 1.22 | 1.85 | 0.71 | 2.38 | 2.25 | 0.25 |
| Green Pea  | Cooked          | 95.64 | 0.36 | 23.91 | 2.97 | 0.89 | 1.32 | 4.27  | 0.92  | 0.97 | 1.21 | 0.27 | 1.11 | 0.25 | 1.82  | 0.72  | 1.19 | 0.69 | 0.67 | 2.23 | 2.13 | 0.23 |
| Green Pea  | Baked           | 97.81 | 2.12 | 22.77 | 3.08 | 0.86 | 1.36 | 4.13  | 0.87  | 0.96 | 1.18 | 0.25 | 1.15 | 0.22 | 2.05  | 0.61  | 1.13 | 1.61 | 0.69 | 2.09 | 2.13 | 0.22 |


| **Samples**        | **ppOPA**     | **ppOPAc**     | **ppTNBS**     | **ppTNBSc**     | **ppKj**       | **3-Enz**      | **4-Enz**      | **AAS x TD**     |
|--------------------|---------------|----------------|----------------|-----------------|----------------|----------------|----------------|------------------|
| **Casein**         | 65.99 ± 2.90  | 88.41 ± 3.88   | 47.04 ± 1.17   | 63.02 ± 1.57    | 87.09 ± 4.09   | 86.05 ± 0.55   | 107.75 ± 1.84  | 91.59 ± 1.67     |
| **Flour (U)**      | 31.65 ± 0.89  | 36.68 ± 1.03   | 26.05 ± 0.74   | 30.19 ± 0.86    | 48.37 ± 2.56   | 53.24 ± 0.21   | 60.40 ± 5.91   | 58.82 ± 1.67     |
| **Flour (H)**      | 39.75 ± 3.57  | 46.07 ± 4.14   | 28.54 ± 1.60   | 33.08 ± 1.85    | 47.69 ± 2.38   | 60.20 ± 0.43   | 61.52 ± 0.25   | 59.06 ± 2.56     |
| **Albumin (U)**    | 37.69 ± 0.83  | 42.90 ± 0.95   | 25.98 ± 0.40   | 29.57 ± 0.46    | 53.84 ± 1.17   | 64.10 ± 0.34   | 73.40 ± 0.80   | 83.43 ± 4.29     |
| **Albumin (H)**    | 47.95 ± 0.44  | 54.58 ± 0.50   | 37.00 ± 0.61   | 42.11 ± 0.70    | 51.12 ± 3.48   | 71.79 ± 0.27   | 77.04 ± 0.32   | 87.42 ± 4.74     |
| **Total globulin (U)** | 18.05 ± 0.50 | 21.15 ± 0.59   | 11.84 ± 0.09   | 13.88 ± 0.19    | 28.58 ± 1.24   | 30.66 ± 0.84   | 35.66 ± 0.23   | 33.08 ± 0.82     |
| **Total globulin (H)** | 26.03 ± 0.77 | 30.49 ± 0.91   | 18.00 ± 0.12   | 21.08 ± 0.59    | 28.60 ± 0.93   | 33.69 ± 0.38   | 36.63 ± 0.18   | 35.95 ± 0.82     |
| **Major globulin (H)** | 20.30 ± 1.29 | 23.71 ± 1.51   | 15.44 ± 0.94   | 18.04 ± 0.94    | 23.32 ± 0.84   | 29.47 ± 0.28   | 29.58 ± 0.05   | 31.23 ± 0.15     |
| **Glutelin (H)**   | 26.32 ± 2.76  | 30.43 ± 3.19   | 26.95 ± 0.90   | 31.15 ± 0.94    | 42.80 ± 2.45   | 60.13 ± 0.31   | 63.88 ± 0.58   | 56.30 ± 1.54     |


| **Correlated methods** | **Correlation coefficients (r)** |
|------------------------|----------------------------------|
| **Pepsin-pancreatin method** |                              |
| ppOPA/ppTNBS           | 0.8172 (p = 0.0132)*             |
| ppOPAc/ppTNBS          | 0.8289 (p = 0.0109)*             |
| ppOPA/ppKj             | 0.6066 (p = 0.1108)              |
| ppOPAc/ppKj            | 0.6380 (p = 0.0887)              |
| ppTNBS/ppKj            | 0.3805 (p = 0.3524)              |
| ppTNBS/ppKj            | 0.4260 (p = 0.2926)              |
| **3-Enzymes/pepsin-pancreatin method** |                  |
| 3-Enz/ppOPA            | 0.6920 (p = 0.0572)              |
| 3-Enz/ppOPAc           | 0.7086 (p = 0.0491)*             |
| 3-Enz/ppTNBS           | 0.8233 (p = 0.0120)*             |
| 3-Enz/ppTNBSc          | 0.8409 (p = 0.0089)*             |
| 3-Enz/ppKj             | 0.6159 (p = 0.1039)              |
| **4-Enzymes/pepsin-pancreatin method** |                  |
| 4-Enz/ppOPA            | 0.5544 (p = 0.1538)              |
| 4-Enz/ppOPAc           | 0.5804 (p = 0.1314)              |
| 4-Enz/ppTNBS           | 0.5240 (p = 0.1824)              |
| 4-Enz/ppTNBSc          | 0.5558 (p = 0.1525)              |
| 4-Enz/ppKj             | 0.8132 (p = 0.0140)*             |
| **4-Enzymes/3-enzymes method** |                           |
| 3-Enz/4-Enz            | 0.8275 (p = 0.0112)*             |
| **In vivo/in vitro**   |                                  |
| TD/ppOPA               | 0.6785 (p = 0.0640)              |
| TD/ppOPAc              | 0.6637 (p = 0.0727)              |
| TD/ppTNBS              | 0.4553 (p = 0.2570)              |
| TD/ppTNBSc             | 0.4481 (p = 0.2654)              |
| TD/ppKj                | 0.2683 (p = 0.5205)              |
| TD/3-Enz               | 0.3534 (p = 0.3904)              |
| TD/4-Enz               | 0.3675 (p = 0.3704)              |


| **Correlated Methods**                    | **Correlation Coefficients (r)** |
|-------------------------------------------|----------------------------------|
| ppOPA/ppTNBS                              | 0.8172 (p = 0.0132)              |
| ppOPAc/ppTNBS                             | 0.8289 (p = 0.0109)              |
| ppOPA/ppKj                                | 0.6066 (p = 0.1108)              |
| ppOPAc/ppKj                               | 0.6380 (p = 0.0887)              |
| ppTNBS/ppKj                               | 0.3805 (p = 0.3524)              |
| ppTNBS/ppKj                               | 0.4260 (p = 0.2926)              |
| 3-Enz/ppOPA                               | 0.6920 (p = 0.0572)              |
| 3-Enz/ppOPAc                              | 0.7086 (p = 0.0491)              |
| 3-Enz/ppTNBS                              | 0.8233 (p = 0.0120)              |
| 3-Enz/ppTNBSc                             | 0.8409 (p = 0.0089)              |
| 3-Enz/ppKj                                | 0.6159 (p = 0.1039)              |
| 4-Enz/ppOPA                               | 0.5544 (p = 0.1538)              |
| 4-Enz/ppOPAc                              | 0.5804 (p = 0.1314)              |
| 4-Enz/ppTNBS                              | 0.5240 (p = 0.1824)              |
| 4-Enz/ppTNBSc                             | 0.5558 (p = 0.1525)              |
| 4-Enz/ppKj                                | 0.8132 (p = 0.0140)              |
| 3-Enz/4-Enz                               | 0.8275 (p = 0.0112)              |
| TD/ppOPA                                  | 0.6785 (p = 0.0640)              |
| TD/ppOPAc                                 | 0.6637 (p = 0.0727)              |
| TD/ppTNBS                                 | 0.4553 (p = 0.2570)              |
| TD/ppTNBSc                                | 0.4481 (p = 0.2654)              |
| TD/ppKj                                   | 0.2683 (p = 0.5205)              |
| TD/3-Enz                                  | 0.3534 (p = 0.3904)              |
| TD/4-Enz                                  | 0.3675 (p = 0.3704)              |


| **Correlated Methods**                    | **Correlation Coefficients (r)** |
|-------------------------------------------|----------------------------------|
| Pepsin-pancreatin method: ppOPA/ppTNBS    | 0.8172 (p = 0.0132)              |
| Pepsin-pancreatin method: ppOPAc/ppTNBS   | 0.8289 (p = 0.0109)              |
| Pepsin-pancreatin method: ppOPA/ppKj      | 0.6066 (p = 0.1108)              |
| Pepsin-pancreatin method: ppOPAc/ppKj     | 0.6380 (p = 0.0887)              |
| Pepsin-pancreatin method: ppTNBS/ppKj     | 0.3805 (p = 0.3524)              |
| Pepsin-pancreatin method: ppTNBS/ppKj     | 0.4260 (p = 0.2926)              |
| 3-Enzymes/pepsin-pancreatin method: 3-Enz/ppOPA | 0.6920 (p = 0.0572)       |
| 3-Enzymes/pepsin-pancreatin method: 3-Enz/ppOPAc| 0.7086 (p = 0.0491)       |
| 3-Enzymes/pepsin-pancreatin method: 3-Enz/ppTNBS| 0.8233 (p = 0.0120)       |
| 3-Enzymes/pepsin-pancreatin method: 3-Enz/ppTNBSc| 0.8409 (p = 0.0089)      |
| 3-Enzymes/pepsin-pancreatin method: 3-Enz/ppKj | 0.6159 (p = 0.1039)       |
| 4-Enzymes/pepsin-pancreatin method: 4-Enz/ppOPA | 0.5544 (p = 0.1538)       |
| 4-Enzymes/pepsin-pancreatin method: 4-Enz/ppOPAc| 0.5804 (p = 0.1314)       |
| 4-Enzymes/pepsin-pancreatin method: 4-Enz/ppTNBS| 0.5240 (p = 0.1824)       |
| 4-Enzymes/pepsin-pancreatin method: 4-Enz/ppTNBSc| 0.5558 (p = 0.1525)      |
| 4-Enzymes/pepsin-pancreatin method: 4-Enz/ppKj | 0.8132 (p = 0.0140)       |
| 4-Enzymes/3-enzymes method: 3-Enz/4-Enz | 0.8275 (p = 0.0112)              |
| In vivo/in vitro: TD/ppOPA                | 0.6785 (p = 0.0640)              |
| In vivo/in vitro: TD/ppOPAc               | 0.6637 (p = 0.0727)              |
| In vivo/in vitro: TD/ppTNBS               | 0.4553 (p = 0.2570)              |
| In vivo/in vitro: TD/ppTNBSc              | 0.4481 (p = 0.2654)              |
| In vivo/in vitro: TD/ppKj                 | 0.2683 (p = 0.5205)              |
| In vivo/in vitro: TD/3-Enz                | 0.3534 (p = 0.3904)              |
| In vivo/in vitro: TD/4-Enz                | 0.3675 (p = 0.3704)              |


In [ ]:
# Reimporting pandas and recreating the data
import pandas as pd

# Creating the updated DataFrame with combined group names
data_combined = {
    "Group (Processing Method)": [
        "Casein",
        "Black (Extruded)", "Black (Cooked)", "Black (Baked)",
        "Faba (Extruded)", "Faba (Cooked)", "Faba (Baked)",
        "Navy (Extruded)", "Navy (Cooked)", "Navy (Baked)",
        "Pinto (Extruded)", "Pinto (Cooked)", "Pinto (Baked)",
        "Red Kidney (Extruded)", "Red Kidney (Cooked)", "Red Kidney (Baked)"
    ],
    "ASP": [7.78, 2.61, 3.11, 2.5, 3.27, 3.04, 3.18, 2.91, 3.04, 2.65, 2.81, 2.73, 2.53, 3.00, 3.16, 3.05],
    "THR": [3.35, 0.98, 1.07, 1.03, 0.98, 0.91, 0.91, 0.99, 1.05, 0.94, 0.83, 0.97, 0.85, 0.99, 1.07, 1.02],
    "SER": [5.64, 1.61, 1.55, 1.48, 1.53, 1.47, 1.49, 1.50, 1.65, 1.44, 1.37, 1.57, 1.31, 1.55, 1.71, 1.62],
    "GLU": [20.05, 3.4, 3.28, 3.06, 4.74, 4.47, 4.59, 3.66, 3.45, 3.25, 3.5, 3.17, 3.22, 3.91, 3.9, 3.89],
    "PRO": [9.77, 0.87, 1.36, 0.94, 1.23, 1.04, 1.24, 0.7, 0.74, 0.73, 0.74, 0.71, 0.6, 0.65, 0.75, 0.78],
    "GLY": [1.35, 0.87, 0.91, 0.78, 1.09, 1.07, 1.09, 0.87, 0.91, 0.82, 0.75, 0.82, 0.78, 0.88, 0.93, 0.94],
    "ALA": [3.16, 0.97, 1.18, 1.08, 1.32, 1.29, 1.34, 1.12, 1.17, 1.05, 1.02, 0.98, 1.04, 1.15, 1.21, 1.25],
    "CYS": [0.78, 0.23, 0.24, 0.25, 0.27, 0.3, 0.32, 0.2, 0.2, 0.21, 0.19, 0.22, 0.19, 0.22, 0.24, 0.23],
    "VAL": [5.02, 1.24, 1.13, 1.05, 1.25, 1.16, 1.26, 1.25, 1.26, 1.16, 1.04, 1.06, 1.06, 1.19, 1.27, 1.22],
    "MET": [1.45, 0.27, 0.23, 0.25, 0.24, 0.21, 0.24, 0.21, 0.22, 0.24, 0.25, 0.27, 0.24, 0.24, 0.24, 0.24],
    "ILE": [3.84, 0.95, 0.91, 0.81, 1.05, 0.99, 1.03, 0.99, 1.05, 0.91, 0.86, 1.03, 0.82, 0.96, 1.03, 1.04],
    "LEU": [8.39, 2.02, 1.83, 1.8, 2.19, 2.19, 2.13, 1.94, 2.12, 1.86, 1.97, 2.02, 1.75, 2.0, 2.21, 2.1],
    "TYR": [4.83, 0.58, 0.82, 0.07, 0.9, 0.8, 0.78, 0.64, 0.68, 0.6, 0.62, 0.7, 0.65, 0.66, 0.72, 0.68],
    "PHE": [4.59, 1.38, 1.47, 1.24, 1.26, 1.12, 1.18, 1.25, 1.42, 1.11, 1.34, 1.38, 1.09, 1.31, 1.43, 1.36],
    "HIS": [2.74, 0.83, 0.9, 0.8, 0.88, 0.83, 0.82, 0.84, 0.87, 0.79, 0.82, 0.83, 0.76, 0.9, 0.96, 0.91],
    "LYS": [6.96, 1.36, 1.68, 1.3, 1.6, 1.77, 1.82, 1.43, 1.67, 1.28, 1.3, 1.62, 1.34, 1.46, 1.8, 1.45],
    "ARG": [3.12, 1.29, 1.53, 1.43, 2.7, 2.74, 2.97, 1.43, 1.44, 1.32, 1.19, 1.37, 1.37, 1.43, 1.49, 1.47],
    "TRP": [1.08, 0.25, 0.28, 0.25, 0.28, 0.19, 0.25, 0.27, 0.3, 0.28, 0.25, 0.25, 0.28, 0.26, 0.3, 0.28],
}

df_combined = pd.DataFrame(data_combined)




In [ ]:
df_combined

,Group (Processing Method),ASP,THR,SER,GLU,PRO,GLY,ALA,CYS,VAL,MET,ILE,LEU,TYR,PHE,HIS,LYS,ARG,TRP
0,Casein,7.78,3.35,5.64,20.05,9.77,1.35,3.16,0.78,5.02,1.45,3.84,8.39,4.83,4.59,2.74,6.96,3.12,1.08
1,Black (Extruded),2.61,0.98,1.61,3.40,0.87,0.87,0.97,0.23,1.24,0.27,0.95,2.02,0.58,1.38,0.83,1.36,1.29,0.25
2,Black (Cooked),3.11,1.07,1.55,3.28,1.36,0.91,1.18,0.24,1.13,0.23,0.91,1.83,0.82,1.47,0.90,1.68,1.53,0.28
3,Black (Baked),2.50,1.03,1.48,3.06,0.94,0.78,1.08,0.25,1.05,0.25,0.81,1.80,0.07,1.24,0.80,1.30,1.43,0.25
4,Faba (Extruded),3.27,0.98,1.53,4.74,1.23,1.09,1.32,0.27,1.25,0.24,1.05,2.19,0.90,1.26,0.88,1.60,2.70,0.28
5,Faba (Cooked),3.04,0.91,1.47,4.47,1.04,1.07,1.29,0.30,1.16,0.21,0.99,2.19,0.80,1.12,0.83,1.77,2.74,0.19
6,Faba (Baked),3.18,0.91,1.49,4.59,1.24,1.09,1.34,0.32,1.26,0.24,1.03,2.13,0.78,1.18,0.82,1.82,2.97,0.25
7,Navy (Extruded),2.91,0.99,1.50,3.66,0.70,0.87,1.12,0.20,1.25,0.21,0.99,1.94,0.64,1.25,0.84,1.43,1.43,0.27
8,Navy (Cooked),3.04,1.05,1.65,3.45,0.74,0.91,1.17,0.20,1.26,0.22,1.05,2.12,0.68,1.42,0.87,1.67,1.44,0.30
9,Navy (Baked),2.65,0.94,1.44,3.25,0.73,0.82,1.05,0.21,1.16,0.24,0.91,1.86,0.60,1.11,0.79,1.28,1.32,0.28


In [ ]:
# Combining MET and CYS into a new column "MET + CYS", and PHE and TYR into "PHE + TYR"
df_combined["MET + CYS"] = df_combined["MET"] + df_combined["CYS"]
df_combined["PHE + TYR"] = df_combined["PHE"] + df_combined["TYR"]

# Dropping the individual MET, CYS, PHE, and TYR columns for simplicity
df_combined_final = df_combined.drop(columns=["MET", "CYS", "PHE", "TYR"])

# Display the updated table
df_combined_final.head(), df_combined_final.columns


(  Group (Processing Method)   ASP   THR   SER    GLU   PRO   GLY   ALA   VAL  \
 0                    Casein  7.78  3.35  5.64  20.05  9.77  1.35  3.16  5.02   
 1          Black (Extruded)  2.61  0.98  1.61   3.40  0.87  0.87  0.97  1.24   
 2            Black (Cooked)  3.11  1.07  1.55   3.28  1.36  0.91  1.18  1.13   
 3             Black (Baked)  2.50  1.03  1.48   3.06  0.94  0.78  1.08  1.05   
 4           Faba (Extruded)  3.27  0.98  1.53   4.74  1.23  1.09  1.32  1.25   
 
     ILE   LEU   HIS   LYS   ARG   TRP  MET + CYS  PHE + TYR  
 0  3.84  8.39  2.74  6.96  3.12  1.08       2.23       9.42  
 1  0.95  2.02  0.83  1.36  1.29  0.25       0.50       1.96  
 2  0.91  1.83  0.90  1.68  1.53  0.28       0.47       2.29  
 3  0.81  1.80  0.80  1.30  1.43  0.25       0.50       1.31  
 4  1.05  2.19  0.88  1.60  2.70  0.28       0.51       2.16  ,
 Index(['Group (Processing Method)', 'ASP', 'THR', 'SER', 'GLU', 'PRO', 'GLY',
        'ALA', 'VAL', 'ILE', 'LEU', 'HIS', 'LYS', 'ARG

In [ ]:
df_combined_final

,Group (Processing Method),ASP,THR,SER,GLU,PRO,GLY,ALA,VAL,ILE,LEU,HIS,LYS,ARG,TRP,MET + CYS,PHE + TYR
0,Casein,7.78,3.35,5.64,20.05,9.77,1.35,3.16,5.02,3.84,8.39,2.74,6.96,3.12,1.08,2.23,9.42
1,Black (Extruded),2.61,0.98,1.61,3.40,0.87,0.87,0.97,1.24,0.95,2.02,0.83,1.36,1.29,0.25,0.50,1.96
2,Black (Cooked),3.11,1.07,1.55,3.28,1.36,0.91,1.18,1.13,0.91,1.83,0.90,1.68,1.53,0.28,0.47,2.29
3,Black (Baked),2.50,1.03,1.48,3.06,0.94,0.78,1.08,1.05,0.81,1.80,0.80,1.30,1.43,0.25,0.50,1.31
4,Faba (Extruded),3.27,0.98,1.53,4.74,1.23,1.09,1.32,1.25,1.05,2.19,0.88,1.60,2.70,0.28,0.51,2.16
5,Faba (Cooked),3.04,0.91,1.47,4.47,1.04,1.07,1.29,1.16,0.99,2.19,0.83,1.77,2.74,0.19,0.51,1.92
6,Faba (Baked),3.18,0.91,1.49,4.59,1.24,1.09,1.34,1.26,1.03,2.13,0.82,1.82,2.97,0.25,0.56,1.96
7,Navy (Extruded),2.91,0.99,1.50,3.66,0.70,0.87,1.12,1.25,0.99,1.94,0.84,1.43,1.43,0.27,0.41,1.89
8,Navy (Cooked),3.04,1.05,1.65,3.45,0.74,0.91,1.17,1.26,1.05,2.12,0.87,1.67,1.44,0.30,0.42,2.10
9,Navy (Baked),2.65,0.94,1.44,3.25,0.73,0.82,1.05,1.16,0.91,1.86,0.79,1.28,1.32,0.28,0.45,1.71


In [ ]:
!pip install pytesseract
!pip install pillow

In [ ]:
import pytesseract
import os
# Configure pytesseract to point to the location of the Tesseract executable
# Replace with the correct path for your OS
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'  # or the relevant path on your system
# Check if the path was set correctly.
print(pytesseract.pytesseract.tesseract_cmd)
print(os.environ.get("TESSDATA_PREFIX"))

/usr/bin/tesseract
None


In [ ]:
def clean_and_divide(value):
    """
    Check if a value contains a '%' sign. If yes, remove the '%' and divide by 100.
    Otherwise, convert to a float and return unchanged.
    """
    value = str(value).strip()  # Ensure value is a string and strip whitespace
    if '%' in value:
        return float(value.replace('%', '')) / 100  # Remove '%' and divide by 100
    return float(value)  # Convert to float directly if no '%'



In [ ]:
# Step 1: Load data into DataFrames
df_pdcaas = pd.DataFrame(pdcaas_data)
df_ivpdcaas = pd.DataFrame(ivpdcaas_data)

# Step 2: Rename columns explicitly to align both tables
df_ivpdcaas = df_ivpdcaas.rename(columns={"Crude Protein": "CP"})

# Step 3: Filter relevant columns
df_pdcaas_filtered = df_pdcaas[["Protein", "CP", "PDCAAS"]]
df_ivpdcaas_filtered = df_ivpdcaas[["Protein", "CP", "IVPD", "TPD"]]

# Step 4: Merge the tables on "Protein" and "CP"
merged_df = pd.merge(df_pdcaas_filtered, df_ivpdcaas_filtered, on=["Protein", "CP"], how="inner")

# Step 5: Display the final merged table
print("Final Merged Table:")
print(merged_df)

NameError: name 'pdcaas_data' is not defined

In [ ]:
column_to_delete = "DIAAS"
column_to_merge_1 = "PDCAAS"
column_to_merge_2 = "IVPDCAAS"
column_to_merge_3 = "TPD"
column_to_merge_4 = "IVPD"


if column_to_delete in df.columns:
    df = df.drop(columns=[column_to_delete])
    print(f"Deleted column: {column_to_delete}")

# Step 2: Check and merge two columns if both exist
if column_to_merge_1 in df.columns and column_to_merge_2 in df.columns:
    # Merge the two columns (taking the average here as an example)
    df[merged_column_name] = df[[column_to_merge_1, column_to_merge_2]].sum(axis=1)

    # Drop the original columns after merging
    # df = df.drop(columns=[column_to_merge_1, column_to_merge_2])
    # print(f"Merged columns: '{column_to_merge_1}' and '{column_to_merge_2}' into '{merged_column_name}'")

# Display the final DataFrame
print("Final DataFrame:")
print(df)

In [ ]:
# Create DataFrames
df_amino_acids = pd.DataFrame(data1) # this data frame contains the data about amino acids
df_scores = pd.DataFrame(data2) # this dataframe contains the data about socres like pdcaas, ivpdcaas etc.

# Step 1: Merge the two tables on the 'Food' column
merged_df = pd.merge(df_amino_acids, df_scores, on="Food", how="inner") # mergin two tables bases on the common column luke food column

# Step 2: Display the merged table
print("Merged Table:")
print(merged_df)